# ⚡ Time-to-Failure Prediction — XGBoost (Statistical Features)

**Dataset:** LANL Earthquake Prediction (laboratory acoustic emission signals)
**Goal:** Predict `time_to_failure` (seconds until the next simulated quake) from engineered
statistical descriptors of the raw `acoustic_data` signal.
**Model:** `XGBRegressor`

---

## Scientific Motivation

In stick-slip laboratory friction experiments — the physical system underlying the LANL dataset —
a rock sample is sheared under constant load until it slips, simulating a single earthquake cycle in
miniature. As the fault interface approaches failure, micro-fracturing intensifies and the acoustic
emission signal grows measurably more volatile: higher variance, more extreme amplitude spikes,
heavier tails. This notebook tests whether a small set of *hand-engineered* statistical descriptors —
rather than the full raw waveform — is sufficient for a gradient-boosted tree ensemble to track that
precursor signature and regress `time_to_failure` accurately. Notebook 3 then asks the complementary
question: can a 1D-CNN do *better* by learning its own representations end-to-end from the raw signal?

### Notebook outline
1. Imports & configuration
2. Stream raw signal & build the segment-level feature table
3. Feature engineering recap
4. Exploratory data analysis
5. Train / test split (80 / 20)
6. Train XGBoost
7. Held-out evaluation
8. Feature importance
9. Persist the trained model
10. Result summary


## 1 · Imports & Configuration

In [ ]:
import os
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sys.path.append(os.path.abspath(os.path.join("..", "src")))
from config import (
    LANL_TRAIN_PATH, LANL_FEATURES_PATH, LANL_STAT_FEATURE_COLUMNS,
    SEGMENT_SIZE, LANL_TEST_SIZE, XGB_PARAMS, RANDOM_STATE,
    MODELS_DIR, FIGURES_DIR, PLOT_COLORS, PLOT_DPI, PLOT_STYLE,
)
from logging_utils import console, get_logger, metrics_table, section
from preprocessing import build_segment_feature_table

logger = get_logger(__name__)

# Shared dark, high-contrast publication theme (identical to src/train_xgboost.py)
plt.style.use(PLOT_STYLE)
sns.set_style("darkgrid", {"axes.facecolor": "#1b1f27", "grid.color": PLOT_COLORS["grid"]})
plt.rcParams.update({
    "figure.dpi": PLOT_DPI,
    "savefig.dpi": PLOT_DPI,
    "axes.edgecolor": PLOT_COLORS["text"],
    "axes.labelcolor": PLOT_COLORS["text"],
    "text.color": PLOT_COLORS["text"],
    "xtick.color": PLOT_COLORS["text"],
    "ytick.color": PLOT_COLORS["text"],
    "font.size": 11,
})

console.print(f"[bold]Segment size:[/bold] {SEGMENT_SIZE:,} samples  |  [bold]Random seed:[/bold] {RANDOM_STATE}")

## 2 · Stream the Raw Signal & Build Segment Features

The raw LANL `train.csv` is several gigabytes — far too large to load into memory in one shot. We
stream it in chunks aligned to the project's `SEGMENT_SIZE` (150,000 rows), carrying any "leftover"
rows across chunk boundaries so a segment is never split mid-window. The engineered feature table is
cached to `data/processed/` so repeat runs skip re-processing the raw signal entirely.

In [ ]:
section("Prepare Segment-Level Statistical Features")

_READER_DTYPES = {"acoustic_data": np.int16, "time_to_failure": np.float64}


def load_or_build_features() -> pd.DataFrame:
    if os.path.exists(LANL_FEATURES_PATH):
        logger.info("Found cached segment features at '%s'.", LANL_FEATURES_PATH)
        return pd.read_csv(LANL_FEATURES_PATH)

    logger.info("No cache found -- streaming raw signal from '%s'.", LANL_TRAIN_PATH)
    reader = pd.read_csv(LANL_TRAIN_PATH, dtype=_READER_DTYPES, chunksize=SEGMENT_SIZE)

    feature_rows = []
    leftover_acoustic = np.array([], dtype=np.int16)
    leftover_ttf = np.array([], dtype=np.float64)

    for chunk in reader:
        acoustic = np.concatenate([leftover_acoustic, chunk["acoustic_data"].values])
        ttf = np.concatenate([leftover_ttf, chunk["time_to_failure"].values])

        n_full = len(acoustic) // SEGMENT_SIZE
        usable = n_full * SEGMENT_SIZE

        if n_full > 0:
            # build_segment_feature_table renders its own tqdm progress bar
            # over segments within this chunk
            feature_rows.append(
                build_segment_feature_table(acoustic[:usable], ttf[:usable], segment_size=SEGMENT_SIZE)
            )

        leftover_acoustic = acoustic[usable:]
        leftover_ttf = ttf[usable:]

    features_df = pd.concat(feature_rows, ignore_index=True)
    os.makedirs(os.path.dirname(LANL_FEATURES_PATH), exist_ok=True)
    features_df.to_csv(LANL_FEATURES_PATH, index=False)
    logger.info("Cached %d segments to '%s'.", len(features_df), LANL_FEATURES_PATH)
    return features_df


features_df = load_or_build_features()
console.print(f"[green]Feature table ready[/green]: {features_df.shape[0]:,} segments x {features_df.shape[1]} columns.")
features_df.head()

## 3 · Feature Engineering Recap

For every 150,000-sample window, `preprocessing.extract_segment_features` computes **7 statistical
descriptors**:

| Feature | Description | Seismological role |
|---|---|---|
| `mean` | Average signal amplitude | Baseline acoustic activity level |
| `std` | Signal volatility | Rises as micro-fracturing intensifies pre-failure |
| `max` | Peak positive amplitude | Captures large discrete acoustic events |
| `min` | Peak negative amplitude | Captures large discrete acoustic events (opposite phase) |
| `median` | Robust central tendency | Less sensitive to spike outliers than `mean` |
| `p01` | 1st percentile | Lower-tail / negative-spike behavior |
| `p99` | 99th percentile | Upper-tail / positive-spike behavior — often the strongest precursor signal |

Tail percentiles and extrema specifically capture the precursor micro-fracture "spikes" that a plain
`mean`/`std` summary would smooth over — this is the central feature-engineering insight that lets a
gradient-boosted tree model approximate the information a deep network would otherwise have to learn
on its own from the raw waveform.

In [ ]:
features_df.describe()

## 4 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(features_df["time_to_failure"], bins=40, kde=True, ax=axes[0], color=PLOT_COLORS["secondary"])
axes[0].set_title("Distribution of time_to_failure", fontweight="bold")
axes[0].set_xlabel("time_to_failure (s)")
axes[0].set_ylabel("Segment count")
axes[0].grid(alpha=0.3)

sns.scatterplot(
    data=features_df, x="std", y="time_to_failure",
    alpha=0.5, ax=axes[1], color=PLOT_COLORS["primary"], edgecolor="none",
)
axes[1].set_title("Signal Volatility (std) vs. time_to_failure", fontweight="bold")
axes[1].set_xlabel("Segment std (acoustic amplitude)")
axes[1].set_ylabel("time_to_failure (s)")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "xgb_eda.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

**Reading the relationship:** segments with elevated `std` cluster toward lower `time_to_failure`
values — consistent with the stick-slip failure model, where acoustic volatility ramps up as the
simulated fault approaches its slip point.

## 5 · Train / Test Split (80 / 20)

In [ ]:
X = features_df[LANL_STAT_FEATURE_COLUMNS]
y = features_df["time_to_failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=LANL_TEST_SIZE, random_state=RANDOM_STATE
)

console.print(f"Train: [bold]{X_train.shape}[/bold]  |  Test: [bold]{X_test.shape}[/bold]")

## 6 · Train XGBoost

Hyperparameters are pulled from the project-wide `config.XGB_PARAMS`:
`n_estimators=100`, `learning_rate=0.1`, `max_depth=5`.

In [ ]:
section("Train XGBoost")
console.print(f"Hyperparameters: {XGB_PARAMS}")

model = xgb.XGBRegressor(**XGB_PARAMS)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False,
)

evals_result = model.evals_result()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(evals_result["validation_0"]["rmse"], label="Train RMSE", color=PLOT_COLORS["primary"], linewidth=2)
ax.plot(evals_result["validation_1"]["rmse"], label="Test RMSE", color=PLOT_COLORS["secondary"], linewidth=2)
ax.set_xlabel("Boosting Round")
ax.set_ylabel("RMSE")
ax.set_title("XGBoost Training Curve", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "xgb_training_curve.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

## 7 · Evaluation on Held-Out Test Set

In [ ]:
section("Evaluation on Held-Out Test Set")

y_pred = model.predict(X_test)

metrics = {
    "mae": mean_absolute_error(y_test, y_pred),
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "r2": r2_score(y_test, y_pred),
}
console.print(metrics_table("XGBoost -- Test Set Metrics", metrics))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred, alpha=0.45, s=14, color=PLOT_COLORS["secondary"], edgecolors="none")
lims = [y_test.min(), y_test.max()]
ax.plot(lims, lims, "--", linewidth=1.8, color=PLOT_COLORS["accent"], label="Perfect prediction")
ax.set_xlabel("Actual time_to_failure (s)")
ax.set_ylabel("Predicted time_to_failure (s)")
ax.set_title("XGBoost: Actual vs. Predicted Time-to-Failure", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "xgb_actual_vs_predicted.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

## 8 · Feature Importance

In [ ]:
importance = pd.Series(
    model.feature_importances_, index=LANL_STAT_FEATURE_COLUMNS
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=importance.values, y=importance.index, hue=importance.index,
            palette="flare", legend=False, edgecolor="none", ax=ax)
ax.set_title("XGBoost Feature Importance", fontweight="bold")
ax.set_xlabel("Importance")
ax.set_ylabel("")
ax.grid(alpha=0.3, axis="x")
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "xgb_feature_importance.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

console.print(metrics_table("Feature Importance", importance.to_dict()))

**Interpretation:** tail percentiles (`p99`, `p01`) and `std` typically rank above `mean` —
confirming the seismological hypothesis that *spike behavior*, not baseline amplitude, carries the
strongest precursor signal for an approaching labquake.

## 9 · Persist the Trained Model

In [ ]:
section("Persist Model Artifact")

os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, "xgboost_lanl.joblib")
joblib.dump(model, model_path)
console.print(f"[bold green]Model saved[/bold green] -> {model_path}")

---
## 10 · Result Summary

| Metric | Value |
|---|---|
| MAE | *(see Section 7 metrics table above)* |
| Hyperparameters | `n_estimators=100`, `learning_rate=0.1`, `max_depth=5` |
| Train / Test Split | 80 / 20 |

The statistical-feature approach achieves strong accuracy with a lightweight, fast-to-train model —
well suited for resource-constrained or real-time monitoring deployments. Notebook 3 explores whether
a deep learning model can extract richer signal representations directly from the raw waveform,
forgoing hand-engineered features entirely.
